# LeetCode 3270 — Find the Key of the Numbers bằng Python

Link đề: https://leetcode.com/problems/find-the-key-of-the-numbers/ · Độ khó: **Easy** · Chủ đề: Math, String (xử lý chữ số)

Skeleton LeetCode:

```python
class Solution:
    def generateKey(self, num1: int, num2: int, num3: int) -> int:
        ...
```

Ghi chú: ba ví dụ chính lấy từ đề; những ví dụ tự dựng thêm có ghi rõ "tự dựng" — nếu muốn khớp từng chữ thì nên đối chiếu lại đề gốc. Mọi đáp án trong notebook đã được kiểm chứng bằng code (kể cả đối chiếu vét cạn toàn bộ các bộ ba nhỏ và hàng chục nghìn bộ ba ngẫu nhiên).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: với ba số nguyên dương `num1`, `num2`, `num3`, **khóa (key)** là một số **bốn chữ số**: trước hết **đệm chữ số `0` vào bên trái** cho đủ bốn chữ số; chữ số thứ `i` của khóa là **chữ số nhỏ nhất** trong ba chữ số thứ `i` của ba số; cuối cùng trả khóa **bỏ các số `0` đứng đầu**.
2. Nắm các công cụ Python: **`str`/`int`** (đổi qua lại, `int("0007") == 7`), **`zfill`** và định dạng **`f"{n:04d}"`** để đệm số `0`, **`zip`** để ghép ba chuỗi theo từng vị trí, **`min`**, **`"".join`**, và **`//`, `%`, `divmod`** để tách chữ số bằng toán.
3. Đi từ cách **đệm chuỗi rồi duyệt theo chỉ số** → **`zip` + `min` + `join`** → **tách chữ số bằng `// 10^i % 10` (không cần chuỗi)** → **tách từ chữ số hàng đơn vị và dừng sớm** → **một biểu thức `sum`**.
4. Hiểu mấu chốt: đây là bài **xử lý chữ số theo từng vị trí**; việc **"đệm `0`"** chính là việc **căn thẳng hàng theo chữ số hàng đơn vị** — thiếu bước này, các chữ số của số ngắn bị ghép nhầm với số dài. Trên toán học, chữ số thứ `i` của một số ngắn hơn đơn giản là `0`, và phép `//`, `%` cho ra `0` một cách tự nhiên.
5. Tránh các lỗi kinh điển: **quên đệm `0`** (`zip` cắt theo chuỗi ngắn nhất và ghép sai vị trí), **so sánh cả số** thay vì từng chữ số, **trả về chuỗi** thay vì số nguyên, quên `% 10` khi tách chữ số, và **đảo thứ tự** các chữ số khi ráp ngược lại.

## 2. Đề bài & ví dụ

Cho ba **số nguyên dương** `num1`, `num2` và `num3`.

**Khóa (key)** của `num1`, `num2`, `num3` được định nghĩa là một số có **bốn chữ số** sao cho:

- Ban đầu, nếu số nào có **ít hơn bốn chữ số** thì được **đệm thêm số `0` ở đầu** cho đủ bốn chữ số.
- Chữ số thứ `i` (`1 <= i <= 4`) của khóa được tạo bằng cách lấy **chữ số nhỏ nhất** trong các chữ số thứ `i` của `num1`, `num2` và `num3`.

Hãy trả về khóa của ba số **không có các số `0` đứng đầu** (nếu có).

**Ví dụ 1**

- Input: `num1 = 1`, `num2 = 10`, `num3 = 1000`
- Output: `0`
- Giải thích: sau khi đệm, `num1 = "0001"`, `num2 = "0010"`, `num3 = "1000"`. Chữ số thứ 1 của khóa là `min(0, 0, 1) = 0`; thứ 2 là `min(0, 0, 0) = 0`; thứ 3 là `min(0, 1, 0) = 0`; thứ 4 là `min(1, 0, 0) = 0`. Khóa là `"0000"` tức `0`.

**Ví dụ 2**

- Input: `num1 = 987`, `num2 = 879`, `num3 = 798`
- Output: `777`

**Ví dụ 3**

- Input: `num1 = 1`, `num2 = 2`, `num3 = 3`
- Output: `1`

**Ràng buộc:** `1 <= num1, num2, num3 <= 9999`.

In [1]:
from typing import List
import random
from itertools import product

examples = [
    (1, 10, 1000, 0),
    (987, 879, 798, 777),
    (1, 2, 3, 1),
]
for a, b, c, expected in examples:
    print(f"num1={a}, num2={b}, num3={c} -> kỳ vọng {expected}")

num1=1, num2=10, num3=1000 -> kỳ vọng 0
num1=987, num2=879, num3=798 -> kỳ vọng 777
num1=1, num2=2, num3=3 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Số và chuỗi: `str(n)` và `int(s)`

`str(987)` cho chuỗi `"987"`; `int("987")` cho số `987`. Chuỗi cho phép nhìn **từng chữ số** (`"987"[0] == "9"`); số cho phép làm toán. Đặc biệt `int("0007")` bằng `7` — **số `0` đứng đầu tự biến mất** khi đổi sang số, đúng với yêu cầu "không có số `0` đứng đầu" của đề.

In [2]:
print(str(987), type(str(987)))          # 987 <class 'str'>
print(int("987"), type(int("987")))      # 987 <class 'int'>
print("987"[0], "987"[2])                # 9 7
print(int("0007"), int("0000"), int("0420"))    # 7 0 420

987 <class 'str'>
987 <class 'int'>
9 7
7 0 420


### 3.2. Đệm số `0` bên trái: `zfill` và `f"{n:04d}"`

`s.zfill(4)` đệm `0` vào **bên trái** cho đủ độ rộng `4` (không đổi nếu đã đủ hoặc dài hơn). `f"{n:04d}"` làm điều tương tự với một **số** `n`: `04d` nghĩa là "rộng `4`, đệm bằng `0`, kiểu số nguyên". `str(n).rjust(4, "0")` cũng được. Đây chính là bước "đệm" của đề.

In [3]:
print("7".zfill(4), "987".zfill(4), "1000".zfill(4))        # 0007 0987 1000
print(f"{7:04d}", f"{987:04d}")                              # 0007 0987
print("42".rjust(4, "0"))                                    # 0042

0007 0987 1000
0007 0987
0042


### 3.3. `zip` ghép các chuỗi theo từng vị trí

`zip(a, b, c)` ghép phần tử cùng vị trí của ba dãy thành các bộ ba: `zip("abc", "xyz", "123")` cho `('a','x','1')`, `('b','y','2')`, `('c','z','3')`. **Chú ý:** `zip` **dừng ở dãy ngắn nhất** — nếu độ dài khác nhau, phần dư của dãy dài bị bỏ. Vì vậy phải **đệm cho cùng độ dài** trước khi `zip` (xem Lỗi 1).

In [4]:
print(list(zip("abc", "xyz", "123")))     # [('a','x','1'), ('b','y','2'), ('c','z','3')]
print(list(zip("ab", "wxyz")))            # [('a','w'), ('b','x')] -> bị cắt theo dãy ngắn nhất

[('a', 'x', '1'), ('b', 'y', '2'), ('c', 'z', '3')]
[('a', 'w'), ('b', 'x')]


### 3.4. `min` trên ba chữ số (kiểu chuỗi hoặc số)

`min(x, y, z)` hoặc `min(bộ_ba)` cho giá trị nhỏ nhất. Với chuỗi **một ký tự chữ số**, so sánh theo mã ký tự cũng đúng thứ tự số: `'0' < '1' < ... < '9'`. Vì vậy `min('9', '8', '7')` là `'7'`. `"".join(...)` ghép nhiều chuỗi thành một.

In [5]:
print(min(9, 8, 7), min("9", "8", "7"))               # 7 7
print(min(("0", "0", "1")))                            # 0
print("".join(["0", "7", "7", "7"]))                  # 0777
print(int("".join(["0", "7", "7", "7"])))             # 777

7 7
0
0777
777


### 3.5. Tách chữ số bằng toán: `//` và `%`

`n % 10` là **chữ số hàng đơn vị**. `n // 10` **bỏ** chữ số hàng đơn vị. Để lấy chữ số ở hàng `10^p` (`p = 0` đơn vị, `1` chục, `2` trăm, `3` nghìn): `n // 10**p % 10`. Với số ngắn hơn, các hàng "cao hơn" tự cho `0` (ví dụ `7 // 100 % 10 = 0`) — **chính là việc đệm `0` mà không cần chuỗi**.

In [6]:
n = 9876
for p in range(4):
    print(f"hàng 10^{p}: {n} // {10 ** p} % 10 =", n // 10 ** p % 10)     # 6, 7, 8, 9
print("số ngắn:", 7 // 10 % 10, 7 // 100 % 10, 7 // 1000 % 10)           # 0 0 0

hàng 10^0: 9876 // 1 % 10 = 6
hàng 10^1: 9876 // 10 % 10 = 7
hàng 10^2: 9876 // 100 % 10 = 8
hàng 10^3: 9876 // 1000 % 10 = 9
số ngắn: 0 0 0


### 3.6. `divmod` và ghép lại chữ số

`divmod(n, 10)` trả `(n // 10, n % 10)` cùng lúc. Để **ráp** các chữ số lại thành số, mỗi chữ số ở hàng `10^p` đóng góp `chữ_số * 10**p`; tổng các đóng góp là số cần tìm. Ví dụ chữ số `(7, 7, 7, 0)` từ hàng đơn vị lên nghìn: `7*1 + 7*10 + 7*100 + 0*1000 = 777`.

In [7]:
print(divmod(987, 10))                     # (98, 7)
digits_low_to_high = [7, 7, 7, 0]          # từ hàng đơn vị lên hàng nghìn
print(sum(d * 10 ** p for p, d in enumerate(digits_low_to_high)))      # 777

(98, 7)
777


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Làm Ví dụ 2 bằng tay: `987`, `879`, `798`. Viết ba số **căn thẳng hàng** (đệm `0` bên trái):

```
  0 9 8 7
  0 8 7 9
  0 7 9 8
 ---------
  0 7 7 7     <- từng cột lấy số nhỏ nhất
```

- Cột 1 (hàng nghìn): `min(0, 0, 0) = 0`.
- Cột 2 (hàng trăm): `min(9, 8, 7) = 7`.
- Cột 3 (hàng chục): `min(8, 7, 9) = 7`.
- Cột 4 (hàng đơn vị): `min(7, 9, 8) = 7`.

Khóa là `0777`; bỏ `0` đầu được `777`. 

Ví dụ 1: `1`, `10`, `1000` thẳng hàng:

```
  0 0 0 1
  0 0 1 0
  1 0 0 0
 ---------
  0 0 0 0   -> 0
```

**Nhận xét:** bài là **"tìm min theo cột"**. Cái bẫy duy nhất là **căn hàng**: số `1` phải đứng ở **cột hàng đơn vị**, không phải ở cột đầu tiên bên trái.

In [8]:
a, b, c = "987".zfill(4), "879".zfill(4), "798".zfill(4)
print(a, b, c)
for col, (x, y, z) in enumerate(zip(a, b, c), start=1):
    print(f"cột {col}: ({x}, {y}, {z}) -> min = {min(x, y, z)}")

0987 0879 0798
cột 1: (0, 0, 0) -> min = 0
cột 2: (9, 8, 7) -> min = 7
cột 3: (8, 7, 9) -> min = 7
cột 4: (7, 9, 8) -> min = 7


## 5. Cách 1 — Brute Force / baseline: đệm chuỗi rồi duyệt theo chỉ số

**Ý tưởng.** Làm đúng từng bước của đề: đổi ba số thành chuỗi, `zfill(4)`; với `i` từ `0` đến `3` lấy `s1[i]`, `s2[i]`, `s3[i]`, tìm min, nối vào chuỗi kết quả; cuối cùng `int(...)`.

In [9]:
def generate_key_brute(num1: int, num2: int, num3: int) -> int:
    s1 = str(num1).zfill(4)
    s2 = str(num2).zfill(4)
    s3 = str(num3).zfill(4)
    key = ""
    for i in range(4):
        d = min(s1[i], s2[i], s3[i])      # chữ số nhỏ nhất ở cột i (so sánh ký tự '0'..'9')
        key += d
    return int(key)                       # int() tự bỏ các số 0 đứng đầu


for a, b, c, expected in examples:
    print(a, b, c, "->", generate_key_brute(a, b, c), "(kỳ vọng", expected, ")")

1 10 1000 -> 0 (kỳ vọng 0 )
987 879 798 -> 777 (kỳ vọng 777 )
1 2 3 -> 1 (kỳ vọng 1 )


**Độ phức tạp.** Luôn đúng `4` cột nên `O(1)` time và space (hằng số — độ dài khóa cố định).

**Nút thắt?** Không có nút thắt về tốc độ; "cải tiến" ở các cách sau là về **độ gọn** và **kỹ thuật** (tách chữ số bằng toán thay vì chuỗi, dừng sớm, biểu thức một dòng), rất hữu ích cho các bài chữ số khác.

## 6. Ý tưởng thuật toán / bước đột phá

Hai cách nhìn bổ sung cho nhau:

1. **Góc nhìn chuỗi:** căn hàng = **đệm `0` trái**; sau đó mọi thứ là xử lý **cột**: `zip` ba chuỗi cho ra từng bộ ba chữ số cùng cột, `min` mỗi bộ, `join` thành chuỗi, `int`.
2. **Góc nhìn toán:** chữ số ở hàng `10^p` của `n` là `n // 10**p % 10`. Với số ngắn, công thức này **tự cho `0`** nên **không cần đệm**. Khóa = `Σ min(chữ số hàng p của ba số) * 10**p` với `p = 0..3`.

Vì khóa luôn có đúng `4` chữ số và mỗi bước `O(1)`, tổng là `O(1)`. Ta sẽ thử cả hai góc nhìn, kèm hai biến thể: dừng sớm khi một số hết chữ số, và một biểu thức `sum`.

In [10]:
# Hai góc nhìn cho cùng một đáp án, trên 987, 879, 798
a, b, c = 987, 879, 798
# Góc nhìn chuỗi
col_min = [min(x, y, z) for x, y, z in zip(str(a).zfill(4), str(b).zfill(4), str(c).zfill(4))]
print(col_min, "->", int("".join(col_min)))
# Góc nhìn toán
total = 0
for p in range(4):
    d = min(a // 10 ** p % 10, b // 10 ** p % 10, c // 10 ** p % 10)
    total += d * 10 ** p
print(total)

['0', '7', '7', '7'] -> 777
777


## 7. Các cách tiếp theo

Phần này có **4 cách** nữa (cùng Cách 1 là 5 cách). Mỗi cách: giải thích → code demo → code nộp LeetCode → độ phức tạp.

### Cách 2 — `zip` + `min` + `join` (gọn bằng thư viện chuẩn)

Giải thích từng đoạn:

- `f"{n:04d}"` đệm `0` bên trái cho đủ bốn chữ số (xem 3.2).
- `zip(s1, s2, s3)` cho ba chữ số cùng cột; `min(t)` lấy chữ số nhỏ nhất của bộ ba `t`.
- `"".join(...)` ghép thành chuỗi bốn ký tự; `int(...)` đổi sang số và bỏ `0` đầu.

In [11]:
def generate_key_zip(num1: int, num2: int, num3: int) -> int:
    s = [f"{n:04d}" for n in (num1, num2, num3)]
    return int("".join(min(t) for t in zip(*s)))


for a, b, c, expected in examples:
    assert generate_key_zip(a, b, c) == expected
print("Cách 2 chạy đúng trên các ví dụ")

Cách 2 chạy đúng trên các ví dụ


In [12]:
# Code nộp LeetCode — Cách 2
class Solution:
    def generateKey(self, num1: int, num2: int, num3: int) -> int:
        s = [f"{n:04d}" for n in (num1, num2, num3)]
        return int("".join(min(t) for t in zip(*s)))


for a, b, c, expected in examples:
    assert Solution().generateKey(a, b, c) == expected
print("OK")

OK


**Độ phức tạp Cách 2:** Time `O(1)`, Space `O(1)` (độ dài cố định bằng `4`). Mẹo `zip(*s)`: dấu `*` "mở" danh sách `s` thành ba đối số `s[0], s[1], s[2]` của `zip`.

### Cách 3 — Tách chữ số bằng `// 10^p % 10` (không dùng chuỗi)

Giải thích từng đoạn:

- `k = 1` là `10^p` (`1, 10, 100, 1000` qua bốn vòng).
- Với mỗi `k`, `num // k % 10` là chữ số ở hàng tương ứng; số ngắn hơn cho `0` tự nhiên.
- `x = min(...)` là chữ số của khóa ở hàng đó; đóng góp `x * k` vào kết quả `ans`.
- `k *= 10` chuyển lên hàng kế tiếp.

In [13]:
def generate_key_arith(num1: int, num2: int, num3: int) -> int:
    ans, k = 0, 1
    for _ in range(4):
        x = min(num1 // k % 10, num2 // k % 10, num3 // k % 10)
        ans += x * k
        k *= 10
    return ans


for a, b, c, expected in examples:
    assert generate_key_arith(a, b, c) == expected
print("Cách 3 chạy đúng trên các ví dụ")

Cách 3 chạy đúng trên các ví dụ


In [14]:
# Code nộp LeetCode — Cách 3
class Solution:
    def generateKey(self, num1: int, num2: int, num3: int) -> int:
        ans, k = 0, 1
        for _ in range(4):
            x = min(num1 // k % 10, num2 // k % 10, num3 // k % 10)
            ans += x * k
            k *= 10
        return ans


for a, b, c, expected in examples:
    assert Solution().generateKey(a, b, c) == expected
print("OK")

OK


**Độ phức tạp Cách 3:** Time `O(1)` (đúng `4` vòng), Space `O(1)`. Không tạo chuỗi nào.

### Cách 4 — Lấy chữ số từ hàng đơn vị và **dừng sớm**

Giải thích từng đoạn:

- Lấy chữ số hàng đơn vị của mỗi số bằng `% 10`, rồi **cắt** nó đi bằng `// 10`.
- Chỉ cần lặp **chừng nào cả ba số còn khác `0`**. Khi *một* số đã hết chữ số (`== 0`) thì mọi hàng cao hơn đều có chữ số `0` ở số đó, nên `min` là `0` — các hàng còn lại của khóa là `0`, **không cần cộng thêm**.
- `place` là `10^p` hiện tại; `ans += min(...) * place`.
- Điều kiện `while num1 and num2 and num3` tận dụng việc `0` là "sai". (Vì mỗi số `<= 9999` nên không bao giờ quá `4` vòng.)

In [15]:
def generate_key_early(num1: int, num2: int, num3: int) -> int:
    ans, place = 0, 1
    while num1 and num2 and num3:                 # cả ba còn chữ số
        ans += min(num1 % 10, num2 % 10, num3 % 10) * place
        num1 //= 10
        num2 //= 10
        num3 //= 10
        place *= 10
    return ans


for a, b, c, expected in examples:
    assert generate_key_early(a, b, c) == expected
print("Cách 4 chạy đúng trên các ví dụ")

Cách 4 chạy đúng trên các ví dụ


In [16]:
# Code nộp LeetCode — Cách 4
class Solution:
    def generateKey(self, num1: int, num2: int, num3: int) -> int:
        ans, place = 0, 1
        while num1 and num2 and num3:
            ans += min(num1 % 10, num2 % 10, num3 % 10) * place
            num1 //= 10
            num2 //= 10
            num3 //= 10
            place *= 10
        return ans


for a, b, c, expected in examples:
    assert Solution().generateKey(a, b, c) == expected
print("OK")

OK


**Độ phức tạp Cách 4:** Time `O(1)` (tối đa `4` vòng; theo số chữ số của số ngắn nhất), Space `O(1)`. Điểm hay: nó **không cần biết** độ rộng `4` — đúng cho mọi độ rộng. (Lưu ý: điều kiện dừng dùng *và* vì chỉ cần **một** số hết chữ số là khóa ở các hàng cao hơn bằng `0`.)

### Cách 5 — Một biểu thức `sum` với `divmod`/lũy thừa

Giải thích từng đoạn: gộp Cách 3 thành một biểu thức. `10 ** p` là hàng thứ `p`; `(n // 10 ** p) % 10` lấy chữ số; `min(...)` trên ba số; nhân `10 ** p` rồi `sum` qua `p = 0..3`. Đọc như công thức toán: `key = Σ_{p=0}^{3} min_i(digit_p(num_i)) · 10^p`.

In [17]:
def generate_key_sum(num1: int, num2: int, num3: int) -> int:
    nums = (num1, num2, num3)
    return sum(min(n // 10 ** p % 10 for n in nums) * 10 ** p for p in range(4))


for a, b, c, expected in examples:
    assert generate_key_sum(a, b, c) == expected
print("Cách 5 chạy đúng trên các ví dụ")

Cách 5 chạy đúng trên các ví dụ


In [18]:
# Code nộp LeetCode — Cách 5
class Solution:
    def generateKey(self, num1: int, num2: int, num3: int) -> int:
        nums = (num1, num2, num3)
        return sum(min(n // 10 ** p % 10 for n in nums) * 10 ** p for p in range(4))


for a, b, c, expected in examples:
    assert Solution().generateKey(a, b, c) == expected
print("OK")

OK


**Độ phức tạp Cách 5:** Time `O(1)`, Space `O(1)`. Ngắn gọn nhưng đọc khó hơn với người mới — hãy hiểu Cách 3 trước.

## 8. TRACE từng bước

Ta trace Cách 1 (chuỗi) trên Ví dụ 2; Cách 3 (toán) trên Ví dụ 2 và một ví dụ tự dựng `num1 = 9999`, `num2 = 305`, `num3 = 4071` (tự dựng; đáp án kiểm bằng vét cạn trong cell); và Cách 4 (dừng sớm) trên Ví dụ 1 để thấy nó dừng sau vài vòng.

In [19]:
def trace_string(num1: int, num2: int, num3: int) -> int:
    s1, s2, s3 = (str(n).zfill(4) for n in (num1, num2, num3))
    print(f"num1={num1} -> {s1}, num2={num2} -> {s2}, num3={num3} -> {s3}")
    key = ""
    for i in range(4):
        d = min(s1[i], s2[i], s3[i])
        key += d
        print(f"  cột {i + 1}: ({s1[i]}, {s2[i]}, {s3[i]}) -> min = {d}, key = {key!r}")
    print(f"  int({key!r}) = {int(key)}")
    return int(key)


assert trace_string(987, 879, 798) == 777
assert trace_string(1, 10, 1000) == 0

num1=987 -> 0987, num2=879 -> 0879, num3=798 -> 0798
  cột 1: (0, 0, 0) -> min = 0, key = '0'
  cột 2: (9, 8, 7) -> min = 7, key = '07'
  cột 3: (8, 7, 9) -> min = 7, key = '077'
  cột 4: (7, 9, 8) -> min = 7, key = '0777'
  int('0777') = 777
num1=1 -> 0001, num2=10 -> 0010, num3=1000 -> 1000
  cột 1: (0, 0, 1) -> min = 0, key = '0'
  cột 2: (0, 0, 0) -> min = 0, key = '00'
  cột 3: (0, 1, 0) -> min = 0, key = '000'
  cột 4: (1, 0, 0) -> min = 0, key = '0000'
  int('0000') = 0


In [20]:
def trace_arith(num1: int, num2: int, num3: int) -> int:
    print(f"num1={num1}, num2={num2}, num3={num3}")
    ans, k = 0, 1
    for step in range(1, 5):
        d1, d2, d3 = num1 // k % 10, num2 // k % 10, num3 // k % 10
        x = min(d1, d2, d3)
        ans += x * k
        print(f"  hàng {k:>4}: chữ số ({d1}, {d2}, {d3}) -> min = {x}, cộng {x * k:>4} -> ans = {ans}")
        k *= 10
    return ans


assert trace_arith(987, 879, 798) == 777
mine = trace_arith(9999, 305, 4071)
assert mine == generate_key_brute(9999, 305, 4071)
print("ví dụ tự dựng cho khóa", mine)

num1=987, num2=879, num3=798
  hàng    1: chữ số (7, 9, 8) -> min = 7, cộng    7 -> ans = 7
  hàng   10: chữ số (8, 7, 9) -> min = 7, cộng   70 -> ans = 77
  hàng  100: chữ số (9, 8, 7) -> min = 7, cộng  700 -> ans = 777
  hàng 1000: chữ số (0, 0, 0) -> min = 0, cộng    0 -> ans = 777
num1=9999, num2=305, num3=4071
  hàng    1: chữ số (9, 5, 1) -> min = 1, cộng    1 -> ans = 1
  hàng   10: chữ số (9, 0, 7) -> min = 0, cộng    0 -> ans = 1
  hàng  100: chữ số (9, 3, 0) -> min = 0, cộng    0 -> ans = 1
  hàng 1000: chữ số (9, 0, 4) -> min = 0, cộng    0 -> ans = 1
ví dụ tự dựng cho khóa 1


In [21]:
def trace_early(num1: int, num2: int, num3: int) -> int:
    print(f"num1={num1}, num2={num2}, num3={num3}")
    ans, place = 0, 1
    loops = 0
    while num1 and num2 and num3:
        loops += 1
        d = min(num1 % 10, num2 % 10, num3 % 10)
        print(f"  vòng {loops}: ({num1}, {num2}, {num3}) hàng đơn vị -> min = {d}, place = {place}")
        ans += d * place
        num1 //= 10
        num2 //= 10
        num3 //= 10
        place *= 10
    print(f"  dừng vì một số đã về 0: ({num1}, {num2}, {num3}); ans = {ans}")
    return ans


assert trace_early(1, 10, 1000) == 0
assert trace_early(987, 879, 798) == 777

num1=1, num2=10, num3=1000
  vòng 1: (1, 10, 1000) hàng đơn vị -> min = 0, place = 1
  dừng vì một số đã về 0: (0, 1, 100); ans = 0
num1=987, num2=879, num3=798
  vòng 1: (987, 879, 798) hàng đơn vị -> min = 7, place = 1
  vòng 2: (98, 87, 79) hàng đơn vị -> min = 7, place = 10
  vòng 3: (9, 8, 7) hàng đơn vị -> min = 7, place = 100
  dừng vì một số đã về 0: (0, 0, 0); ans = 777


Bảng trace Cách 3 với Ví dụ 2 (`987`, `879`, `798`):

| hàng `k` | chữ số của `987` | của `879` | của `798` | `min` | cộng `min * k` | `ans` |
|---|---|---|---|---|---|---|
| 1 | 7 | 9 | 8 | 7 | 7 | 7 |
| 10 | 8 | 7 | 9 | 7 | 70 | 77 |
| 100 | 9 | 8 | 7 | 7 | 700 | 777 |
| 1000 | 0 | 0 | 0 | 0 | 0 | 777 |

Chữ số của khóa được tạo từ **hàng đơn vị lên**, mỗi chữ số nhân đúng `k` rồi cộng — không cần đảo thứ tự. Ở hàng `1000` cả ba số đều "hết chữ số" nên chữ số là `0`, `ans` giữ nguyên.

## 9. Code nộp LeetCode

Bản nộp dùng **Cách 3** (tách chữ số bằng `//` và `%`): không chuỗi, ngắn, ít lỗi.

In [22]:
class Solution:
    def generateKey(self, num1: int, num2: int, num3: int) -> int:
        ans, k = 0, 1                        # k = 10^p: đang xét hàng thứ p
        for _ in range(4):                   # đúng 4 chữ số
            x = min(num1 // k % 10, num2 // k % 10, num3 // k % 10)   # chữ số nhỏ nhất của hàng đó
            ans += x * k                     # đặt vào đúng hàng
            k *= 10
        return ans


sol = Solution()
for a, b, c, expected in examples:
    print(a, b, c, "->", sol.generateKey(a, b, c), "(kỳ vọng", expected, ")")

1 10 1000 -> 0 (kỳ vọng 0 )
987 879 798 -> 777 (kỳ vọng 777 )
1 2 3 -> 1 (kỳ vọng 1 )


## 10. Vì sao thuật toán đúng

- **Công thức chữ số.** Chữ số ở hàng `10^p` của số nguyên không âm `n` là `⌊n / 10^p⌋ mod 10`. Nếu `n < 10^p` thì `⌊n / 10^p⌋ = 0` nên chữ số là `0` — đúng với quy ước "đệm `0` ở đầu". Nghĩa là thuật toán **ngầm đệm** các số ngắn.
- **Từng hàng độc lập.** Đề định nghĩa chữ số `i` của khóa chỉ phụ thuộc vào chữ số `i` của ba số. Ta xét `p = 0, 1, 2, 3` (tương ứng chữ số thứ `4, 3, 2, 1` tính từ trái), mỗi hàng lấy `min` của đúng ba chữ số cùng hàng.
- **Ráp đúng giá trị.** Khóa bốn chữ số có chữ số `d_p` ở hàng `10^p` nên có giá trị `Σ d_p · 10^p` — chính là `ans`. Giá trị số đã **không** có `0` đứng đầu (một số nguyên không lưu `0` đầu), đúng yêu cầu.
- **Đủ bốn hàng.** Vì `num <= 9999` nên không có chữ số ở hàng `10^4` trở lên; vòng lặp bốn lần là đủ.
- **Đối chiếu độc lập.** Cách 1 (đệm chuỗi theo đúng chữ của đề) được so với các cách còn lại trên toàn bộ các bộ ba nhỏ và hàng chục nghìn bộ ba ngẫu nhiên ở mục 12.

## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Đệm chuỗi + duyệt chỉ số | `O(1)` | `O(1)` |
| 2. `zip` + `min` + `join` | `O(1)` | `O(1)` |
| 3. `// 10^p % 10` | `O(1)` | `O(1)` |
| 4. Từ hàng đơn vị, dừng sớm | `O(1)` | `O(1)` |
| 5. Một biểu thức `sum` | `O(1)` | `O(1)` |

Vì độ dài khóa **cố định bằng `4`** nên mọi cách đều `O(1)`. Nếu tổng quát hóa cho số có `L` chữ số và `m` số đầu vào thì là `O(L · m)` time.

## 12. Bộ test

Gồm: ví dụ đề, biên (ba số bằng nhau, giá trị `1` và `9999`, có `0` ở giữa, các số có độ dài khác nhau), **vét cạn mọi bộ ba** trong khoảng nhỏ, đối chiếu cả `5` cách và ngẫu nhiên có seed đúng ràng buộc (`1 <= num <= 9999`).

In [23]:
IMPLS = {
    "brute": generate_key_brute,
    "zip": generate_key_zip,
    "arith": generate_key_arith,
    "early": generate_key_early,
    "sum": generate_key_sum,
    "Solution": Solution().generateKey,
}

# 1) ví dụ đề
for a, b, c, expected in examples:
    for name, f in IMPLS.items():
        assert f(a, b, c) == expected, (name, a, b, c)

# 2) biên và tự dựng (đáp án tính tay)
manual = [
    (1, 1, 1, 1),                    # ba số bằng nhau -> chính nó
    (9999, 9999, 9999, 9999),
    (1234, 1234, 1234, 1234),
    (9999, 1, 9999, 1),              # số 1 kéo chữ số hàng đơn vị về 1, các hàng cao về 0
    (1000, 9999, 9999, 1000),        # chỉ giữ chữ số nghìn là 1, các hàng sau về 0
    (1050, 2305, 4999, 1000),        # tự dựng: cột nghìn min(1,2,4)=1; trăm min(0,3,9)=0; chục min(5,0,9)=0; đơn vị min(0,5,9)=0
    (9, 90, 900, 0),                 # lệch hàng: mọi cột có một số bằng 0
    (4321, 4312, 4333, 4311),        # tự dựng: cột đơn vị min(1,2,3)=1; chục min(2,1,3)=1; trăm 3; nghìn 4
]
for a, b, c, expected in manual:
    for name, f in IMPLS.items():
        assert f(a, b, c) == expected, (name, a, b, c, f(a, b, c), expected)
print("ví dụ đề + biên: PASS")

ví dụ đề + biên: PASS


In [24]:
# 3) vét cạn mọi bộ ba trong khoảng nhỏ và với các "số đặc biệt" ở độ dài khác nhau
special = [1, 2, 9, 10, 11, 99, 100, 101, 999, 1000, 1001, 9998, 9999, 305, 4071, 5050]
count = 0
for a, b, c in product(special, repeat=3):
    expected = generate_key_brute(a, b, c)
    for name, f in IMPLS.items():
        assert f(a, b, c) == expected, (name, a, b, c)
    count += 1
for a, b, c in product(range(1, 25), repeat=3):
    expected = generate_key_brute(a, b, c)
    assert generate_key_arith(a, b, c) == expected and generate_key_early(a, b, c) == expected
    count += 1
print(f"{count} bộ ba vét cạn: PASS")

# 4) ngẫu nhiên có seed, đúng ràng buộc, mọi cách phải đồng nhất
rng = random.Random(3270)
for _ in range(20000):
    a, b, c = (rng.randint(1, 10 ** rng.randint(1, 4) - 1) for _ in range(3))
    a, b, c = (max(1, min(9999, v)) for v in (a, b, c))
    expected = generate_key_brute(a, b, c)
    for name, f in IMPLS.items():
        assert f(a, b, c) == expected, (name, a, b, c)
print("20000 test ngẫu nhiên: tất cả các cách đồng nhất — PASS")

# 5) tính chất: khóa luôn <= mỗi số đầu vào (vì từng chữ số của khóa <= chữ số tương ứng) và đối xứng theo thứ tự đối số
for _ in range(2000):
    a, b, c = (rng.randint(1, 9999) for _ in range(3))
    key = generate_key_arith(a, b, c)
    assert key <= min(a, b, c)
    assert key == generate_key_arith(c, a, b) == generate_key_arith(b, c, a)
print("tính chất key <= min và hoán vị: PASS")

17920 bộ ba vét cạn: PASS


20000 test ngẫu nhiên: tất cả các cách đồng nhất — PASS
tính chất key <= min và hoán vị: PASS


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên đệm `0` — `zip` cắt theo chuỗi ngắn nhất và ghép sai cột

Nếu không `zfill(4)`, `zip("1", "10", "1000")` chỉ cho **một** bộ ba `('1', '1', '1')` (cắt theo chuỗi ngắn nhất, và các ký tự bị ghép **căn trái**, không phải căn theo hàng đơn vị). Kết quả sai.

In [25]:
def sai_khong_dem(num1, num2, num3):
    s1, s2, s3 = str(num1), str(num2), str(num3)     # SAI: chưa zfill(4)
    return int("".join(min(t) for t in zip(s1, s2, s3)))

print("sai:", sai_khong_dem(1, 10, 1000), "(đúng là 0)")
print("sai:", sai_khong_dem(9999, 305, 4071), "(đúng là", generate_key_brute(9999, 305, 4071), ")")
print("đúng:", generate_key_zip(1, 10, 1000), generate_key_zip(9999, 305, 4071))

sai: 1 (đúng là 0)
sai: 305 (đúng là 1 )
đúng: 0 1


### Lỗi 2: So sánh cả số thay vì từng chữ số

`min(num1, num2, num3)` là số nhỏ nhất, **không** phải khóa chữ số-theo-cột. Ví dụ `987, 879, 798` cho `798`, trong khi đáp án đúng là `777`.

In [26]:
def sai_min_ca_so(num1, num2, num3):
    return min(num1, num2, num3)         # SAI: phải xét từng cột

print("sai:", sai_min_ca_so(987, 879, 798), "(đúng là 777)")

sai: 798 (đúng là 777)


### Lỗi 3: Trả về **chuỗi** thay vì số nguyên

Đề yêu cầu số nguyên **không có `0` đứng đầu**. Trả `"0777"` (chuỗi) sai kiểu; chuyển bằng `int(...)` mới bỏ được `0` đầu.

In [27]:
def sai_tra_chuoi(num1, num2, num3):
    s = [f"{n:04d}" for n in (num1, num2, num3)]
    return "".join(min(t) for t in zip(*s))          # SAI: trả chuỗi "0777"

r = sai_tra_chuoi(987, 879, 798)
print("sai:", repr(r), type(r), "| bằng 777?", r == 777)
print("đúng:", repr(generate_key_zip(987, 879, 798)))

sai: '0777' <class 'str'> | bằng 777? False
đúng: 777


### Lỗi 4: Quên `% 10` khi tách chữ số

`num // k` **không** phải là một chữ số: nó là **toàn bộ phần còn lại** sau khi bỏ `k` hàng thấp. Chữ số phải là `num // k % 10`. Thiếu `% 10` thì `min` so sánh cả các "đuôi số" thay vì từng chữ số.

In [28]:
def sai_thieu_mod(num1, num2, num3):
    ans, k = 0, 1
    for _ in range(4):
        x = min(num1 // k, num2 // k, num3 // k)      # SAI: thiếu % 10
        ans += x * k
        k *= 10
    return ans

print("sai:", sai_thieu_mod(987, 879, 798), "(đúng là 777)")
print("sai:", sai_thieu_mod(1234, 1234, 1234), "(đúng là 1234)")

sai: 2288 (đúng là 777)
sai: 4664 (đúng là 1234)


### Lỗi 5: Đảo thứ tự khi ráp lại các chữ số

Khi tách từ hàng đơn vị **lên**, chữ số tìm được **đầu tiên** là chữ số **cuối cùng** của khóa. Nếu ráp bằng `ans = ans * 10 + x` thì khóa bị **đảo ngược**. Dùng `ans += x * k` (với `k` tăng dần) hoặc đảo lại cuối cùng. Với `987/879/798` lỗi hiện ra là `7770` (số `0` đầu bị đẩy xuống cuối); với ví dụ **không đối xứng** như `1234` chữ số bị đảo hẳn thành `4321`.

In [29]:
def sai_dao_thu_tu(num1, num2, num3):
    ans = 0
    for _ in range(4):
        x = min(num1 % 10, num2 % 10, num3 % 10)
        ans = ans * 10 + x                 # SAI: chữ số đầu tìm được là chữ số CUỐI của khóa
        num1, num2, num3 = num1 // 10, num2 // 10, num3 // 10
    return ans

print("sai:", sai_dao_thu_tu(987, 879, 798), "(đúng là 777: số 0 đầu bị đẩy xuống cuối thành 7770)")
print("sai:", sai_dao_thu_tu(1234, 1239, 1235), "(đúng là", generate_key_brute(1234, 1239, 1235), ")")

sai: 7770 (đúng là 777: số 0 đầu bị đẩy xuống cuối thành 7770)
sai: 4321 (đúng là 1234 )


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Xử lý từng chữ số / từng cột"** ⇒ hai đường: **chuỗi** (`str`, `zfill`, `zip`) hoặc **toán** (`% 10`, `// 10`). Chọn chuỗi khi cần đệm/căn trái; chọn toán khi muốn tránh chuyển kiểu.
- **Căn hàng bằng hàng đơn vị**: số ngắn hơn được coi là có `0` ở các hàng cao. `zfill` làm việc này cho chuỗi; `// 10^p % 10` làm cho số.
- **Lấy chữ số hàng `p`:** `n // 10**p % 10`. Thuộc lòng công thức này, nó xuất hiện ở bài "cộng hai số", "đảo số", "tổng chữ số"...
- **`zip` dừng ở dãy ngắn nhất:** luôn đảm bảo các dãy cùng độ dài (đệm trước) hoặc dùng `itertools.zip_longest` với `fillvalue`.
- **`int(...)` dọn `0` đầu:** đổi chuỗi sang số là cách nhanh nhất bỏ `0` thừa; ngược lại trả chuỗi là lỗi kiểu.
- **Test thêm ví dụ không đối xứng:** kiểu `1234` (các chữ số đều khác nhau) để bắt lỗi đảo thứ tự chữ số mà ví dụ đối xứng có thể che khuất.

## 15. Biến thể của bài

1. **`n` số thay vì ba:** `min` trên `n` chữ số cùng hàng; dùng `zip(*list_of_strings)`.
2. **Độ rộng `w` bất kỳ** (không cố định `4`): đệm theo `w` hoặc theo độ dài số lớn nhất.
3. **Lấy chữ số lớn nhất** (`max`) thay cho nhỏ nhất.
4. **Phép toán khác theo chữ số:** tổng chữ số cùng hàng `mod 10`, hiệu, hoặc "chữ số xuất hiện nhiều nhất".
5. **Phép AND theo bit:** ở hệ nhị phân, `min` từng bit chính là phép **AND** — bài này là "AND theo từng chữ số thập phân".

In [30]:
# Minh họa biến thể 1 (n số), 2 (độ rộng w) và 3 (max)
def key_n(nums: List[int], w: int = 4, pick=min) -> int:
    strs = [str(n).zfill(w) for n in nums]
    return int("".join(pick(col) for col in zip(*strs)))


print(key_n([987, 879, 798]))                    # 777
print(key_n([5, 50, 500, 5000], w=4))            # 0
print(key_n([12345, 12399, 12355], w=5))         # 12345
print(key_n([987, 879, 798], pick=max))          # 999 (max từng cột: 0,9,9,9)

777
0
12345
999


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đệm chuỗi + chỉ số | Làm đúng từng bước đề | `O(1)` | `O(1)` | Hiểu đề; bản chuẩn đối chiếu |
| 2. `zip` + `min` + `join` | Ghép cột bằng `zip(*s)` | `O(1)` | `O(1)` | Ngắn gọn khi quen `zip` / chuỗi |
| 3. `// 10^p % 10` | Tách chữ số bằng toán | `O(1)` | `O(1)` | **Cách nộp chuẩn**, không cần chuỗi |
| 4. Từ hàng đơn vị, dừng sớm | `% 10` rồi `// 10` đến khi một số về `0` | `O(1)` | `O(1)` | Khi độ rộng không cố định |
| 5. Một biểu thức `sum` | Công thức `Σ min(...)·10^p` | `O(1)` | `O(1)` | Khi muốn code cô đọng |

Lời khuyên: nộp **Cách 3**; nhớ **Cách 2** cho các bài cần đệm/căn chuỗi, và **Cách 4** cho bài độ rộng tùy ý.

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"7".zfill(4)`, `1234 // 10 % 10`, `int("0420")`, `Solution().generateKey(5, 50, 500)`.

**Bài 2 (tính tay).** `num1 = 4321`, `num2 = 56`, `num3 = 7890`: viết ba số thẳng hàng sau khi đệm, lấy min từng cột, rồi cho khóa (không có `0` đứng đầu).

**Bài 3 (điền TODO).** Hoàn thành `generate_key_todo(num1, num2, num3)` theo Cách 3 (`// k % 10`).

In [31]:
def generate_key_todo(num1, num2, num3):
    # TODO 1: ans = 0, k = 1
    # TODO 2: lặp 4 lần: x = min(num1 // k % 10, num2 // k % 10, num3 // k % 10)
    # TODO 3: ans += x * k rồi k *= 10
    # TODO 4: trả về ans
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Không nhìn lời giải, viết `generate_key_n(nums)` nhận **một danh sách** các số nguyên dương (mỗi số `<= 9999`) và trả về khóa bốn chữ số của chúng (không `0` đầu), dùng `zfill` + `zip(*...)`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `generate_key_max(num1, num2, num3)` lấy chữ số **lớn nhất** mỗi cột. (b) Viết `generate_key_any_width(a, b, c)` đệm theo độ dài của số **dài nhất** (không cố định `4`). (c) Viết `count_keys_zero(limit)` đếm số bộ ba `(a, b, c)` với `1 <= a, b, c <= limit` mà khóa bằng `0`; kiểm tra với vét cạn nhỏ `limit = 12`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"7".zfill(4) = "0007"`; `1234 // 10 = 123`, `123 % 10 = 3`; `int("0420") = 420`; `5, 50, 500` thẳng hàng là `0005`, `0050`, `0500` — mỗi cột có một số `0` nên khóa là `0`. Bài 2: đệm `56` thành `0056`.
- **Mức 2 (vừa):** Bài 3: giống mục 9. Bài 4: `strs = [str(n).zfill(4) for n in nums]`, rồi `zip(*strs)` cho từng cột.
- **Mức 3 (sâu):** Bài 5(a): đổi `min` thành `max`. 5(b): `w = max(len(str(n)) for n in (a, b, c))` rồi `zfill(w)`. 5(c): dùng `product(range(1, limit + 1), repeat=3)`; có thể tăng tốc bằng đếm theo logic, nhưng vét cạn `12^3 = 1728` bộ là đủ.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert "7".zfill(4) == "0007"
assert 1234 // 10 % 10 == 3
assert int("0420") == 420
assert Solution().generateKey(5, 50, 500) == 0

# Bài 2: 4321 / 0056 / 7890 -> cột nghìn min(4,0,7)=0; trăm min(3,0,8)=0; chục min(2,5,9)=2; đơn vị min(1,6,0)=0 -> "0020" = 20
assert [f"{n:04d}" for n in (4321, 56, 7890)] == ["4321", "0056", "7890"]
assert Solution().generateKey(4321, 56, 7890) == 20


# Bài 3
def generate_key_todo_ans(num1, num2, num3):
    ans, k = 0, 1
    for _ in range(4):
        x = min(num1 // k % 10, num2 // k % 10, num3 // k % 10)
        ans += x * k
        k *= 10
    return ans


for a, b, c, expected in examples:
    assert generate_key_todo_ans(a, b, c) == expected


# Bài 4
def generate_key_n(nums):
    strs = [str(n).zfill(4) for n in nums]
    return int("".join(min(col) for col in zip(*strs)))


assert generate_key_n([987, 879, 798]) == 777
assert generate_key_n([1, 10, 1000]) == 0
assert generate_key_n([1234]) == 1234
rng = random.Random(11)
for _ in range(500):
    a, b, c = (rng.randint(1, 9999) for _ in range(3))
    assert generate_key_n([a, b, c]) == Solution().generateKey(a, b, c)


# Bài 5
def generate_key_max(num1, num2, num3):
    ans, k = 0, 1
    for _ in range(4):
        ans += max(num1 // k % 10, num2 // k % 10, num3 // k % 10) * k
        k *= 10
    return ans


def generate_key_any_width(a, b, c):
    w = max(len(str(n)) for n in (a, b, c))
    strs = [str(n).zfill(w) for n in (a, b, c)]
    return int("".join(min(col) for col in zip(*strs)))


def count_keys_zero(limit):
    return sum(1 for a, b, c in product(range(1, limit + 1), repeat=3) if generate_key_arith(a, b, c) == 0)


assert generate_key_max(987, 879, 798) == 999          # cột: 0, 9, 9, 9
assert generate_key_max(1, 10, 1000) == 1011           # cột: 1, 0, 1, 1
assert generate_key_any_width(12345, 12399, 12355) == 12345
assert generate_key_any_width(987, 879, 798) == 777
assert generate_key_any_width(1, 10, 1000) == 0
# vét cạn nhỏ: tự đếm không dùng hàm trên
def zero_key_brute(a, b, c):
    for s in zip(*(f"{n:04d}" for n in (a, b, c))):
        if min(s) != "0":
            return False
    return True


assert count_keys_zero(12) == sum(1 for a, b, c in product(range(1, 13), repeat=3) if zero_key_brute(a, b, c))
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tách từng chữ số bằng `// 10^p % 10`, lấy `min`, đặt đúng hàng)

```python
class Solution:
    def generateKey(self, num1: int, num2: int, num3: int) -> int:
        ans, k = 0, 1
        for _ in range(4):
            x = min(num1 // k % 10, num2 // k % 10, num3 // k % 10)
            ans += x * k
            k *= 10
        return ans
```

**Checklist**

- [ ] Có **căn hàng** (đệm `0` bằng `zfill(4)` hoặc dùng `// k % 10` tự cho `0`)?
- [ ] Lấy `min` **từng chữ số cùng hàng**, không `min` cả số?
- [ ] Tách chữ số đúng công thức `n // k % 10` (có `% 10`)?
- [ ] Ráp đúng thứ tự: chữ số hàng `k` đóng góp `x * k` (không đảo)?
- [ ] Trả về **số nguyên** (không `0` đứng đầu), không trả chuỗi?
- [ ] Đã thử ba ví dụ đề (`0`, `777`, `1`) và một ví dụ không đối xứng (như `1234`)?
- [ ] Skeleton đúng `class Solution` / `def generateKey(self, num1: int, num2: int, num3: int) -> int`?